# Titanic Dataset — Exploratory Data Analysis (EDA)
**Elevvo Internship | Task 2 — Level 1**

This notebook explores the Titanic passenger dataset, cleans missing and duplicate data, checks data types, calculates summary statistics, investigates survival patterns by gender and passenger class, and visualizes relationships and correlations.

> **Dataset loading:** The notebook downloads the CSV directly from a public GitHub raw URL. You do not need to upload a local dataset file.

## Objectives
1. Load and inspect the dataset.
2. Understand columns, data types, missing values, and duplicates.
3. Clean the data using appropriate strategies.
4. Generate summary statistics and group-based insights.
5. Visualize survival rates by gender, class, and embarkation port.
6. Explore numerical correlations and summarize key findings.


## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Load the dataset online

The dataset is the commonly used Titanic training dataset (891 passenger records). If the first source is unavailable, the second public source is tried automatically.

In [ ]:
dataset_urls = [
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv",
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
]

df = None
last_error = None

for url in dataset_urls:
    try:
        df = pd.read_csv(url)
        print(f"Dataset loaded successfully from: {url}")
        break
    except Exception as error:
        last_error = error
        print(f"Could not load from {url}")

if df is None:
    raise RuntimeError(
        "The dataset could not be downloaded. Check your internet connection and run this cell again."
    ) from last_error

# Standardize column names so the remaining analysis works with either source.
df.columns = [col.strip().replace(" ", "_") for col in df.columns]

# The fallback seaborn dataset uses lowercase column names and has no PassengerId.
if "survived" in df.columns and "Survived" not in df.columns:
    df = df.rename(columns={
        "survived": "Survived", "pclass": "Pclass", "sex": "Sex",
        "age": "Age", "sibsp": "SibSp", "parch": "Parch",
        "fare": "Fare", "embarked": "Embarked"
    })

print("Dataset shape:", df.shape)
display(df.head())


## 3. Initial data inspection

In [ ]:
print("Dataset information:")
df.info()

print("\\nSummary statistics:")
display(df.describe(include="all").T)

print("\\nMissing values:")
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing Percentage", ascending=False)
display(missing_summary)

print("\\nDuplicate rows:", df.duplicated().sum())


## 4. Data cleaning

Cleaning decisions:
- Remove exact duplicate rows.
- Fill missing `Age` values with the median because age is numeric and can be skewed.
- Fill missing `Embarked` values with the mode (most frequent category).
- Drop `Cabin` because it has a very high proportion of missing values in this dataset.
- Keep rows with missing `Survived` out of the analysis because the target outcome is required.
- Convert relevant columns to appropriate data types.

The original data is preserved in `df_raw`; cleaning is performed on a copy.

In [ ]:
df_raw = df.copy()
df_clean = df.copy()

# Remove exact duplicate records.
df_clean = df_clean.drop_duplicates().copy()

# Drop Cabin when it exists and is mostly missing.
if "Cabin" in df_clean.columns and df_clean["Cabin"].isna().mean() > 0.40:
    df_clean = df_clean.drop(columns=["Cabin"])

# Fill missing numeric and categorical values.
if "Age" in df_clean.columns:
    df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())

if "Embarked" in df_clean.columns:
    embarked_mode = df_clean["Embarked"].mode(dropna=True)
    if not embarked_mode.empty:
        df_clean["Embarked"] = df_clean["Embarked"].fillna(embarked_mode.iloc[0])

# The outcome must be known for survival analysis.
if "Survived" in df_clean.columns:
    df_clean = df_clean.dropna(subset=["Survived"])
    df_clean["Survived"] = df_clean["Survived"].astype("int64")

# Convert categorical columns to category dtype when present.
for col in ["Sex", "Embarked", "Pclass"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype("category")

print("Original shape:", df_raw.shape)
print("Cleaned shape:", df_clean.shape)
print("\\nMissing values after cleaning:")
display(df_clean.isna().sum().to_frame("Missing Count"))


## 5. Feature engineering

In [ ]:
# Create useful features for passenger-level analysis.
if {"SibSp", "Parch"}.issubset(df_clean.columns):
    df_clean["FamilySize"] = (
        df_clean["SibSp"].astype(int) + df_clean["Parch"].astype(int) + 1
    )
    df_clean["IsAlone"] = (df_clean["FamilySize"] == 1).astype(int)

if "Age" in df_clean.columns:
    df_clean["AgeGroup"] = pd.cut(
        df_clean["Age"],
        bins=[0, 12, 18, 35, 60, np.inf],
        labels=["Child", "Teenager", "Young Adult", "Adult", "Senior"],
        include_lowest=True
    )

display(df_clean.head())


## 6. Overall survival statistics

In [ ]:
survival_counts = df_clean["Survived"].value_counts().sort_index()
survival_percentages = df_clean["Survived"].value_counts(normalize=True).sort_index() * 100

survival_summary = pd.DataFrame({
    "Passenger Outcome": ["Did not survive", "Survived"],
    "Count": [survival_counts.get(0, 0), survival_counts.get(1, 0)],
    "Percentage": [survival_percentages.get(0, 0), survival_percentages.get(1, 0)]
})
display(survival_summary.round(2))

ax = sns.barplot(data=survival_summary, x="Passenger Outcome", y="Count")
ax.set_title("Titanic Passenger Survival Counts")
ax.set_xlabel("")
ax.set_ylabel("Number of Passengers")
plt.show()


## 7. Survival by gender

In [ ]:
gender_survival = (
    df_clean.groupby("Sex", observed=False)["Survived"]
    .agg(Passengers="count", Survival_Rate="mean")
    .reset_index()
)
gender_survival["Survival_Rate"] *= 100
display(gender_survival.round(2))

sns.barplot(data=gender_survival, x="Sex", y="Survival_Rate")
plt.title("Survival Rate by Gender")
plt.xlabel("Gender")
plt.ylabel("Survival Rate (%)")
plt.ylim(0, 100)
plt.show()


## 8. Survival by passenger class

In [ ]:
class_survival = (
    df_clean.groupby("Pclass", observed=False)["Survived"]
    .agg(Passengers="count", Survival_Rate="mean")
    .reset_index()
)
class_survival["Survival_Rate"] *= 100
display(class_survival.round(2))

sns.barplot(data=class_survival, x="Pclass", y="Survival_Rate")
plt.title("Survival Rate by Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate (%)")
plt.ylim(0, 100)
plt.show()


## 9. Survival by gender and passenger class

In [ ]:
gender_class_survival = (
    df_clean.groupby(["Pclass", "Sex"], observed=False)["Survived"]
    .mean()
    .mul(100)
    .reset_index(name="Survival_Rate")
)
display(gender_class_survival.round(2))

sns.barplot(
    data=gender_class_survival,
    x="Pclass",
    y="Survival_Rate",
    hue="Sex"
)
plt.title("Survival Rate by Gender and Passenger Class")
plt.xlabel("Passenger Class")
plt.ylabel("Survival Rate (%)")
plt.ylim(0, 100)
plt.legend(title="Gender")
plt.show()


## 10. Survival by embarkation port

In [ ]:
if "Embarked" in df_clean.columns:
    embarked_survival = (
        df_clean.groupby("Embarked", observed=False)["Survived"]
        .agg(Passengers="count", Survival_Rate="mean")
        .reset_index()
    )
    embarked_survival["Survival_Rate"] *= 100
    display(embarked_survival.round(2))

    sns.barplot(data=embarked_survival, x="Embarked", y="Survival_Rate")
    plt.title("Survival Rate by Embarkation Port")
    plt.xlabel("Embarkation Port")
    plt.ylabel("Survival Rate (%)")
    plt.ylim(0, 100)
    plt.show()
else:
    print("Embarked column is not available in this dataset.")


## 11. Age distribution and survival

In [ ]:
sns.histplot(data=df_clean, x="Age", hue="Survived", bins=30, kde=True, multiple="layer")
plt.title("Age Distribution by Survival Outcome")
plt.xlabel("Age")
plt.ylabel("Number of Passengers")
plt.show()

if "AgeGroup" in df_clean.columns:
    age_group_survival = (
        df_clean.groupby("AgeGroup", observed=False)["Survived"]
        .mean().mul(100).reset_index(name="Survival_Rate")
    )
    display(age_group_survival.round(2))

    sns.barplot(data=age_group_survival, x="AgeGroup", y="Survival_Rate")
    plt.title("Survival Rate by Age Group")
    plt.xlabel("Age Group")
    plt.ylabel("Survival Rate (%)")
    plt.xticks(rotation=20)
    plt.ylim(0, 100)
    plt.show()


## 12. Family size and survival

In [ ]:
if "FamilySize" in df_clean.columns:
    family_survival = (
        df_clean.groupby("FamilySize")["Survived"]
        .agg(Passengers="count", Survival_Rate="mean")
        .reset_index()
    )
    family_survival["Survival_Rate"] *= 100
    display(family_survival.round(2))

    sns.barplot(data=family_survival, x="FamilySize", y="Survival_Rate")
    plt.title("Survival Rate by Family Size")
    plt.xlabel("Family Size")
    plt.ylabel("Survival Rate (%)")
    plt.ylim(0, 100)
    plt.show()


## 13. Correlation analysis and heatmap

In [ ]:
numeric_df = df_clean.select_dtypes(include=np.number)
correlation_matrix = numeric_df.corr()

display(correlation_matrix.round(2))

plt.figure(figsize=(10, 7))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()


## 14. Key findings (generated from the data)

In [ ]:
overall_survival_rate = df_clean["Survived"].mean() * 100
print(f"Overall survival rate: {overall_survival_rate:.1f}%")

if not gender_survival.empty:
    best_gender = gender_survival.loc[gender_survival["Survival_Rate"].idxmax()]
    print(
        f"Highest survival rate by gender: {best_gender['Sex']} "
        f"({best_gender['Survival_Rate']:.1f}%)."
    )

if not class_survival.empty:
    best_class = class_survival.loc[class_survival["Survival_Rate"].idxmax()]
    print(
        f"Highest survival rate by passenger class: "
        f"Class {best_class['Pclass']} ({best_class['Survival_Rate']:.1f}%)."
    )

if "Embarked" in df_clean.columns and "embarked_survival" in globals() and not embarked_survival.empty:
    best_port = embarked_survival.loc[embarked_survival["Survival_Rate"].idxmax()]
    print(
        f"Highest survival rate by embarkation port: "
        f"{best_port['Embarked']} ({best_port['Survival_Rate']:.1f}%)."
    )

print("\\nInterpretation reminder:")
print("- These are descriptive patterns, not proof that one feature caused survival.")
print("- Group sizes should be considered when comparing percentages.")
print("- Correlation does not imply causation.")


## 15. Final conclusion

Use the tables and charts above to write a short conclusion in your own words. A good EDA conclusion should mention:

- The overall survival percentage.
- Differences in survival rates by gender and passenger class.
- Any visible age-group or family-size patterns.
- Important relationships shown in the correlation heatmap.
- A limitation: the analysis is observational and does not establish causation.

### Elevvo Task 2 checklist
- [x] Load Titanic dataset using Python.
- [x] Inspect data and generate summary statistics.
- [x] Check and handle missing values.
- [x] Check and remove duplicate rows.
- [x] Check/convert relevant data types.
- [x] Analyze survival by gender and passenger class.
- [x] Visualize key patterns and correlations.
- [x] Add bar plots and a correlation heatmap (bonus).

**Before submission:** Run all cells from top to bottom in Jupyter Notebook or Google Colab, make sure the charts appear, review the generated findings, and save the notebook with its outputs.
